In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np

In [2]:
path = "C:\\Program Files\\MetaTrader 5\\terminal64.exe"
login = 5030595537
password = "-r7lRdVn"
server = "MetaQuotes-Demo"
timeout = 10000
portable = False
if mt5.initialize(path=path, login=login, password=password, server=server, timeout=timeout, portable=portable):
    print("Initialization successful")
else:
    print("Initialization not successful")

Initialization successful


In [3]:
account_info_dict = mt5.account_info()._asdict()
for key, value in account_info_dict.items():
    print(f"{key}: {value}")

login: 5030595537
trade_mode: 0
leverage: 100
limit_orders: 200
margin_so_mode: 0
trade_allowed: True
trade_expert: True
margin_mode: 2
currency_digits: 2
fifo_close: False
balance: 96484.9
credit: 0.0
profit: -7913.0
equity: 88571.9
margin: 24899.92
margin_free: 63671.98
margin_level: 355.7115846155329
margin_so_call: 50.0
margin_so_so: 30.0
margin_initial: 0.0
margin_maintenance: 0.0
assets: 0.0
liabilities: 0.0
commission_blocked: 0.0
name: Praise Godwins
server: MetaQuotes-Demo
currency: USD
company: MetaQuotes Ltd.


In [9]:
symbol = "EURUSD"

timeframe = mt5.TIMEFRAME_M1

In [10]:
from datetime import datetime

import pytz

end_time = datetime.today().astimezone(pytz.utc)

In [11]:
# get hourly data
eurusd_rates = mt5.copy_rates_from(symbol, timeframe, end_time, 10)

In [12]:
eurusd_df = pd.DataFrame(eurusd_rates)
# format time 

eurusd_df['time'] = pd.to_datetime(eurusd_df['time'], unit='s').dt.tz_localize('UTC')
eurusd_df.head()

,time,open,high,low,close,tick_volume,spread,real_volume
0,2024-10-16 09:00:00+00:00,1.08884,1.08887,1.08826,1.08828,90,3,0
1,2024-10-16 09:01:00+00:00,1.08826,1.08844,1.08819,1.08836,55,4,0
2,2024-10-16 09:02:00+00:00,1.08834,1.08836,1.08805,1.08806,47,4,0
3,2024-10-16 09:03:00+00:00,1.08811,1.08819,1.08783,1.08808,63,3,0
4,2024-10-16 09:04:00+00:00,1.08808,1.08822,1.08807,1.08807,48,4,0


In [14]:
df = eurusd_df
df.head()

,time,open,high,low,close,tick_volume,spread,real_volume
0,2024-10-16 09:00:00+00:00,1.08884,1.08887,1.08826,1.08828,90,3,0
1,2024-10-16 09:01:00+00:00,1.08826,1.08844,1.08819,1.08836,55,4,0
2,2024-10-16 09:02:00+00:00,1.08834,1.08836,1.08805,1.08806,47,4,0
3,2024-10-16 09:03:00+00:00,1.08811,1.08819,1.08783,1.08808,63,3,0
4,2024-10-16 09:04:00+00:00,1.08808,1.08822,1.08807,1.08807,48,4,0


In [15]:
from joblib import load

# Load the model from the file
loaded_model = load('exodus.joblib')

C:\Users\ADMIN\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [12:11:30] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\gbm\gbtree.cc:388: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  warnings.warn(smsg, UserWarning)
C:\Users\ADMIN\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [12:11:30] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\context.cc:43: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
C:\Users\ADMIN\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [12:11:30] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


In [30]:
# fetch data for prediction from metatrader

# Define the symbol and timeframe
symbol = "EURUSD"
timeframe = mt5.TIMEFRAME_M1  # Hourly data

# Fetch data
rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, 1000)
rates_frame = pd.DataFrame(rates)

# Convert time in seconds to datetime
rates_frame['time'] = pd.to_datetime(rates_frame['time'], unit='s')

# Prepare data for model prediction
X = rates_frame[['open', 'high', 'low', 'close', 'tick_volume']]

# Duplicate the column
df['Adj Close'] = df['close']

df['return'] = df['Adj Close'] - df['Adj Close'].shift(1)
return_range = df['return'].max() - df['return'].min()
df['return'] = df['return'] / return_range

df['label'] = df['return'].shift(-1)
df['label'] = df['label'].apply(lambda x: 1 if x>0.0 else 0)
df.dropna(inplace=True)

In [35]:

n_features = 9 # number of features

train_x = np.array([]).reshape([-1,n_features])

for index, row in df.iterrows():
    i = df.index.get_loc(index)
    if i<n_features:
        continue

    _x = np.array(df[i-n_features+1:i+1]['return']).T.reshape([1, -1])
    _y = df.iloc[i]['label'] # Use .iloc for positional indexing
    train_x = np.vstack((train_x, _x))

# Make predictions
predictions = loaded_model.predict(train_x)

# Print or process predictions
print(predictions)

ValueError: Feature shape mismatch, expected: 60, got 9

In [32]:
print(train_x.shape)

(0, 60)


In [33]:
df.shape

(9, 11)

In [34]:
df.head()

,time,open,high,low,close,tick_volume,spread,real_volume,Adj Close,return,label
1,2024-10-16 09:01:00+00:00,1.08826,1.08844,1.08819,1.08836,55,4,0,1.08836,0.205128,0
2,2024-10-16 09:02:00+00:00,1.08834,1.08836,1.08805,1.08806,47,4,0,1.08806,-0.769231,1
3,2024-10-16 09:03:00+00:00,1.08811,1.08819,1.08783,1.08808,63,3,0,1.08808,0.051282,0
4,2024-10-16 09:04:00+00:00,1.08808,1.08822,1.08807,1.08807,48,4,0,1.08807,-0.025641,1
5,2024-10-16 09:05:00+00:00,1.08807,1.08821,1.08806,1.08816,46,4,0,1.08816,0.230769,1


In [29]:
df.tail()

,time,open,high,low,close,tick_volume,spread,real_volume,Adj Close,return,label
5,2024-10-16 09:05:00+00:00,1.08807,1.08821,1.08806,1.08816,46,4,0,1.08816,0.230769,1
6,2024-10-16 09:06:00+00:00,1.08816,1.08833,1.08816,1.08825,37,4,0,1.08825,0.230769,0
7,2024-10-16 09:07:00+00:00,1.08824,1.08824,1.08801,1.08805,41,4,0,1.08805,-0.512821,1
8,2024-10-16 09:08:00+00:00,1.08805,1.08818,1.08801,1.08814,37,4,0,1.08814,0.230769,1
9,2024-10-16 09:09:00+00:00,1.08813,1.08830,1.08812,1.08817,45,4,0,1.08817,0.076923,0


In [36]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np

# Initialize connection to MetaTrader 5
if not mt5.initialize():
    print("initialize() failed")
    mt5.shutdown()

# Define the symbol and timeframe
symbol = "EURUSD"
timeframe = mt5.TIMEFRAME_M1  # 1-minute data

# Fetch data
rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, 1000)
rates_frame = pd.DataFrame(rates)

# Convert time in seconds to datetime
rates_frame['time'] = pd.to_datetime(rates_frame['time'], unit='s')

# Prepare data for model prediction
rates_frame['return'] = rates_frame['close'].diff()
return_range = rates_frame['return'].max() - rates_frame['return'].min()
rates_frame['return'] = rates_frame['return'] / return_range

rates_frame['label'] = rates_frame['return'].shift(-1)
rates_frame['label'] = rates_frame['label'].apply(lambda x: 1 if x > 0.0 else 0)
rates_frame.dropna(inplace=True)

# Select the necessary columns for features
X = rates_frame[['open', 'high', 'low', 'close', 'tick_volume']]

# Define number of features (n_features)
n_features = min(60, len(rates_frame) - 1)

# Initialize train_x with shape (0, n_features)
train_x = np.empty((0, n_features))

# Prepare features based on the 'return' column
for i in range(n_features, len(rates_frame)):
    _x = rates_frame['return'].iloc[i-n_features:i].values.reshape(1, -1)
    train_x = np.vstack((train_x, _x))

# Ensure train_x is not empty before making predictions
if train_x.size > 0:
    # Load your trained model (assuming loaded_model is your trained model)
    # loaded_model = ... (load your model here)
    
    # Make predictions
    predictions = loaded_model.predict(train_x)
    
    # Print or process predictions
    print(predictions)
else:
    print("Not enough data to create features with the current n_features value.")

# Shutdown MetaTrader 5 connection
mt5.shutdown()


[0 1 0 1 0 0 0 1 1 1 1 1 0 0 0 0 1 1 1 0 0 1 0 1 0 1 1 1 0 1 1 1 0 0 0 0 0
 0 0 0 1 0 0 1 0 0 0 0 0 1 0 0 0 0 1 1 0 1 1 1 0 0 1 1 1 1 1 0 0 0 1 1 1 0
 1 0 0 0 0 0 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 1
 0 1 0 0 0 1 1 1 1 1 1 0 0 0 1 0 1 1 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0 1 0 0
 0 0 0 1 1 1 0 0 0 0 0 1 0 0 1 0 1 1 1 0 1 0 0 1 0 1 1 0 0 0 1 0 1 0 0 0 0
 0 0 0 0 1 1 0 0 0 0 0 0 0 1 0 0 1 1 0 0 1 1 0 1 1 1 0 1 0 1 1 0 0 0 0 1 1
 0 1 0 0 0 1 0 1 0 0 0 0 1 0 0 1 1 1 1 0 0 1 1 1 1 1 0 1 1 0 0 0 0 0 0 1 0
 1 0 0 0 1 0 1 1 1 1 0 1 0 0 0 1 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 1 0 1 0 0 1
 0 0 0 1 1 0 0 0 0 1 0 1 1 1 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 0 0 0 1 0 1
 0 0 1 0 1 1 0 1 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 1 1 0 0 1 0 1 1 1 0 0 0 0
 1 1 1 1 1 0 0 0 1 1 0 1 0 0 0 0 1 1 1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0
 1 0 0 0 0 1 1 1 1 0 0 1 0 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 1 0 1 0 1 1 1 0 0 0 0 0 0 1 0 1 0
 0 1 0 0 1 0 1 0 0 0 0 1 

True

In [37]:
predictions.shape

(939,)

In [38]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime
import time

# Initialize MetaTrader 5
if not mt5.initialize():
    print("initialize() failed")
    mt5.shutdown()

# Define the symbol and timeframe
symbol = "EURUSD"
timeframe = mt5.TIMEFRAME_M1  # 1-minute data

# Function to fetch and preprocess data
def fetch_and_preprocess_data():
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, 1000)
    rates_frame = pd.DataFrame(rates)
    rates_frame['time'] = pd.to_datetime(rates_frame['time'], unit='s')
    
    rates_frame['return'] = rates_frame['close'].diff()
    return_range = rates_frame['return'].max() - rates_frame['return'].min()
    rates_frame['return'] = rates_frame['return'] / return_range
    
    rates_frame['label'] = rates_frame['return'].shift(-1)
    rates_frame['label'] = rates_frame['label'].apply(lambda x: 1 if x > 0.0 else 0)
    rates_frame.dropna(inplace=True)
    
    return rates_frame

# Function to prepare features
def prepare_features(rates_frame, n_features):
    train_x = np.empty((0, n_features))
    for i in range(n_features, len(rates_frame)):
        _x = rates_frame['return'].iloc[i-n_features:i].values.reshape(1, -1)
        train_x = np.vstack((train_x, _x))
    return train_x

# Function to place a trade
def place_trade(symbol, action, lot=0.1, slippage=5):
    point = mt5.symbol_info(symbol).point
    price = mt5.symbol_info_tick(symbol).ask if action == "BUY" else mt5.symbol_info_tick(symbol).bid
    deviation = slippage
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": mt5.ORDER_TYPE_BUY if action == "BUY" else mt5.ORDER_TYPE_SELL,
        "price": price,
        "deviation": deviation,
        "magic": 234000,
        "comment": "Automated trade",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_RETURN,
    }
    
    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        print(f"Trade failed, retcode={result.retcode}")
    else:
        print(f"Trade successful: {action} {lot} lot of {symbol} at {price}")

# Load your trained model (assuming loaded_model is your trained model)
# loaded_model = ... (load your model here)

# Define number of features (n_features)
n_features = 60  # Adjust based on your model and data

# Main trading loop
while True:
    rates_frame = fetch_and_preprocess_data()
    if len(rates_frame) < n_features:
        print("Not enough data to create features with the current n_features value.")
        time.sleep(60)  # Wait for 1 minute before retrying
        continue
    
    train_x = prepare_features(rates_frame, n_features)
    
    if train_x.size > 0:
        # Make predictions
        predictions = loaded_model.predict(train_x)
        
        # Get the latest prediction
        latest_prediction = predictions[-1]
        
        # Place a trade based on the latest prediction
        if latest_prediction == 1:
            place_trade(symbol, "BUY")
        else:
            place_trade(symbol, "SELL")
    
    # Wait for the next time interval
    time.sleep(60)  # Wait for 1 minute before fetching new data

# Shutdown MetaTrader 5
mt5.shutdown()


Trade successful: BUY 0.1 lot of EURUSD at 1.08957
Trade successful: SELL 0.1 lot of EURUSD at 1.08959
Trade successful: BUY 0.1 lot of EURUSD at 1.08963
Trade successful: SELL 0.1 lot of EURUSD at 1.08955
Trade successful: BUY 0.1 lot of EURUSD at 1.08941
Trade successful: BUY 0.1 lot of EURUSD at 1.08927


KeyboardInterrupt: 

In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime
import time
import joblib
import logging

# Configure logging
logging.basicConfig(filename='trading_new_log.log', level=logging.INFO, format='%(asctime)s %(message)s')

# Initialize MetaTrader 5
if not mt5.initialize():
    logging.error("initialize() failed")
    mt5.shutdown()

# Define the symbol and timeframe
symbol = "EURUSD"
timeframe = mt5.TIMEFRAME_M1  # 1-minute data

# Load your trained model (assuming loaded_model is your trained model)
model_path = "nexo_xgb.joblib"  # Replace with your model's path
loaded_model = joblib.load(model_path)

# Function to fetch and preprocess data
def fetch_and_preprocess_data():
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, 1000)
    rates_frame = pd.DataFrame(rates)
    rates_frame['time'] = pd.to_datetime(rates_frame['time'], unit='s')
    
    rates_frame['return'] = rates_frame['close'].diff()
    return_range = rates_frame['return'].max() - rates_frame['return'].min()
    rates_frame['return'] = rates_frame['return'] / return_range
    
    rates_frame['label'] = rates_frame['return'].shift(-1)
    rates_frame['label'] = rates_frame['label'].apply(lambda x: 1 if x > 0.0 else 0)
    rates_frame.dropna(inplace=True)
    
    return rates_frame

# Function to prepare features
def prepare_features(rates_frame, n_features):
    train_x = np.empty((0, n_features))
    for i in range(n_features, len(rates_frame)):
        _x = rates_frame['return'].iloc[i-n_features:i].values.reshape(1, -1)
        train_x = np.vstack((train_x, _x))
    return train_x

# Function to place a trade
def place_trade(symbol, action, lot=10.0, slippage=5, stop_loss=30, take_profit=10):
    point = mt5.symbol_info(symbol).point
    price = mt5.symbol_info_tick(symbol).ask if action == "BUY" else mt5.symbol_info_tick(symbol).bid
    deviation = slippage
    
    # Calculate stop loss and take profit prices
    if action == "BUY":
        sl_price = price - stop_loss * point
        tp_price = price + take_profit * point
    else:
        sl_price = price + stop_loss * point
        tp_price = price - take_profit * point
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": mt5.ORDER_TYPE_BUY if action == "BUY" else mt5.ORDER_TYPE_SELL,
        "price": price,
        "sl": sl_price,
        "tp": tp_price,
        "deviation": deviation,
        "magic": 234000,
        "comment": "Automated trade",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_RETURN,
    }
    
    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        logging.error(f"Trade failed, retcode={result.retcode}")
    else:
        logging.info(f"Trade successful: {action} {lot} lot of {symbol} at {price}, SL={sl_price}, TP={tp_price}")

# Function to check the number of open trades
def check_open_trades(symbol):
    positions = mt5.positions_get(symbol=symbol)

    if positions:
        # create a list of dictionaries containing the data for each position
        data = pd.DataFrame([position._asdict() for position in positions])
        print("Unrealized P&L: ", data.profit.sum())
        # print the DataFrame
        display(data.head())
    return len(positions)
    

# Main trading loop
n_features = 60  # Adjust based on your model and data
max_open_trades = 5
max_loss_threshold = -1000  # Maximum allowable loss

while True:
    try:
        # Fetch and preprocess data
        rates_frame = fetch_and_preprocess_data()
        if len(rates_frame) < n_features:
            logging.warning("Not enough data to create features with the current n_features value.")
            time.sleep(30)  # Wait for 1 minute before retrying
            continue
        
        # Prepare features
        train_x = prepare_features(rates_frame, n_features)
        
        if train_x.size > 0:
            # Make predictions
            predictions = loaded_model.predict(train_x)
            
            # Get the latest prediction
            latest_prediction = predictions[-1]
            
            # Check the number of open trades
            open_trades = check_open_trades(symbol)
            if open_trades >= max_open_trades:
                logging.warning("Maximum number of open trades reached.")
                time.sleep(30)  # Wait for 1 minute before retrying
                continue
            
            # Check the account equity to ensure we haven't exceeded the max loss threshold
            account_info = mt5.account_info()
            if account_info.equity - account_info.balance <= max_loss_threshold:
                logging.warning("Maximum loss threshold exceeded. Halting trading.")
                break
            
            # Place a trade based on the latest prediction
            if latest_prediction == 0:
                place_trade(symbol, "BUY")
                # pass
            else:
                place_trade(symbol, "SELL")
        
        # Wait for the next time interval
        time.sleep(30)  # Wait for 1 minute before fetching new data
    
    except Exception as e:
        logging.error(f"An error occurred: {e}")
        time.sleep(60)  # Wait for 1 minute before retrying

# Shutdown MetaTrader 5
mt5.shutdown()


Unrealized P&L:  21.1


,ticket,time,time_msc,time_update,time_update_msc,type,magic,identifier,reason,volume,price_open,sl,tp,price_current,swap,profit,symbol,comment,external_id
0,51930262798,1734437705,1734437705622,1734437705,1734437705622,0,0,51930262798,3,0.1,1.04861,0.0,0.0,1.04918,0.0,5.7,EURUSD,,
1,51930319617,1734438668,1734438668194,1734438668,1734438668194,0,0,51930319617,3,0.1,1.04833,0.0,0.0,1.04918,0.0,8.5,EURUSD,,
2,51930395385,1734440036,1734440036058,1734440036,1734440036058,0,0,51930395385,3,0.1,1.04864,0.0,0.0,1.04918,0.0,5.4,EURUSD,,
3,51930398168,1734440097,1734440097233,1734440097,1734440097233,0,0,51930398168,3,0.1,1.04876,0.0,0.0,1.04918,0.0,4.2,EURUSD,,
4,51930524752,1734442348,1734442348244,1734442348,1734442348244,0,0,51930524752,3,0.1,1.04917,0.0,0.0,1.04918,0.0,0.1,EURUSD,,


Unrealized P&L:  17.6


,ticket,time,time_msc,time_update,time_update_msc,type,magic,identifier,reason,volume,price_open,sl,tp,price_current,swap,profit,symbol,comment,external_id
0,51930262798,1734437705,1734437705622,1734437705,1734437705622,0,0,51930262798,3,0.1,1.04861,0.0,0.0,1.04913,0.0,5.2,EURUSD,,
1,51930319617,1734438668,1734438668194,1734438668,1734438668194,0,0,51930319617,3,0.1,1.04833,0.0,0.0,1.04913,0.0,8.0,EURUSD,,
2,51930395385,1734440036,1734440036058,1734440036,1734440036058,0,0,51930395385,3,0.1,1.04864,0.0,0.0,1.04913,0.0,4.9,EURUSD,,
3,51930398168,1734440097,1734440097233,1734440097,1734440097233,0,0,51930398168,3,0.1,1.04876,0.0,0.0,1.04913,0.0,3.7,EURUSD,,
4,51930524752,1734442348,1734442348244,1734442348,1734442348244,0,0,51930524752,3,0.1,1.04917,0.0,0.0,1.04913,0.0,-0.4,EURUSD,,


Unrealized P&L:  21.8


...

,ticket,time,time_msc,time_update,time_update_msc,type,magic,identifier,reason,volume,price_open,sl,tp,price_current,swap,profit,symbol,comment,external_id
0,51930911396,1734448177,1734448177761,1734448177,1734448177761,0,234000,51930911396,3,10.0,1.04973,1.04943,1.04983,1.04958,0.0,-150.0,EURUSD,Automated trade,


Unrealized P&L:  -190.0


,ticket,time,time_msc,time_update,time_update_msc,type,magic,identifier,reason,volume,price_open,sl,tp,price_current,swap,profit,symbol,comment,external_id
0,51930911396,1734448177,1734448177761,1734448177,1734448177761,0,234000,51930911396,3,10.0,1.04973,1.04943,1.04983,1.04959,0.0,-140.0,EURUSD,Automated trade,
1,51930916303,1734448238,1734448238649,1734448238,1734448238649,1,234000,51930916303,3,10.0,1.04958,1.04988,1.04948,1.04963,0.0,-50.0,EURUSD,Automated trade,


: 

: 

In [5]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime
import time
import joblib
import logging
import os

# Configure logging
logging.basicConfig(filename='trading_new_log.log', level=logging.INFO, format='%(asctime)s %(message)s')

# Initialize MetaTrader 5
if not mt5.initialize():
    logging.error("initialize() failed")
    mt5.shutdown()

# Define the symbol and timeframe
symbol = "EURUSD"
timeframe = mt5.TIMEFRAME_M1  # 1-minute data

# Load your trained model (assuming loaded_model is your trained model)
model_path = "exodus.joblib"  # Replace with your model's path
loaded_model = joblib.load(model_path)

# Function to fetch and preprocess data
def fetch_and_preprocess_data():
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, 1000)
    rates_frame = pd.DataFrame(rates)
    rates_frame['time'] = pd.to_datetime(rates_frame['time'], unit='s')
    
    rates_frame['return'] = rates_frame['close'].diff()
    return_range = rates_frame['return'].max() - rates_frame['return'].min()
    rates_frame['return'] = rates_frame['return'] / return_range
    
    rates_frame['label'] = rates_frame['return'].shift(-1)
    rates_frame['label'] = rates_frame['label'].apply(lambda x: 1 if x > 0.0 else 0)
    rates_frame.dropna(inplace=True)
    
    return rates_frame

# Function to prepare features
def prepare_features(rates_frame, n_features):
    train_x = np.empty((0, n_features))
    for i in range(n_features, len(rates_frame)):
        _x = rates_frame['return'].iloc[i-n_features:i].values.reshape(1, -1)
        train_x = np.vstack((train_x, _x))
    return train_x

# Function to log trade outcome to CSV file
def log_trade_to_csv(trade_info):
    file_exists = os.path.isfile('trade_outcomes.csv')
    trade_info_df = pd.DataFrame([trade_info])
    trade_info_df.to_csv('trade_outcomes.csv', mode='a', header=not file_exists, index=False)

# Function to place a trade
def place_trade(symbol, action, lot=10.0, slippage=5, stop_loss=15, take_profit=10):
    point = mt5.symbol_info(symbol).point
    price = mt5.symbol_info_tick(symbol).ask if action == "BUY" else mt5.symbol_info_tick(symbol).bid
    deviation = slippage
    
    # Calculate stop loss and take profit prices
    if action == "BUY":
        sl_price = price - stop_loss * point
        tp_price = price + take_profit * point
    else:
        sl_price = price + stop_loss * point
        tp_price = price - take_profit * point
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": mt5.ORDER_TYPE_BUY if action == "BUY" else mt5.ORDER_TYPE_SELL,
        "price": price,
        "sl": sl_price,
        "tp": tp_price,
        "deviation": deviation,
        "magic": 234000,
        "comment": "Automated trade",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_RETURN,
    }
    
    result = mt5.order_send(request)
    trade_info = {
        'time': datetime.now(),
        'symbol': symbol,
        'action': action,
        'lot': lot,
        'price': price,
        'sl_price': sl_price,
        'tp_price': tp_price,
        'result': 'success' if result.retcode == mt5.TRADE_RETCODE_DONE else 'failure',
        'retcode': result.retcode,
        'profit': None,  # Placeholder for profit
        'volume': None   # Placeholder for volume
    }
    
    if result.retcode == mt5.TRADE_RETCODE_DONE:
        logging.info(f"Trade successful: {action} {lot} lot of {symbol} at {price}, SL={sl_price}, TP={tp_price}")
    else:
        logging.error(f"Trade failed, retcode={result.retcode}")
    
    return result, trade_info

# Function to check the number of open trades and update trade outcome
def check_open_trades(symbol):
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        # create a list of dictionaries containing the data for each position
        data = pd.DataFrame([position._asdict() for position in positions])
        print("Unrealized P&L: ", data.profit.sum())
        return len(positions), data
    return 0, None

# Main trading loop
n_features = 60  # Adjust based on your model and data
max_open_trades = 5
max_loss_threshold = -1000  # Maximum allowable loss
open_trade_infos = []  # List to store open trade information

while True:
    try:
        # Fetch and preprocess data
        rates_frame = fetch_and_preprocess_data()
        if len(rates_frame) < n_features:
            logging.warning("Not enough data to create features with the current n_features value.")
            time.sleep(30)  # Wait for 1 minute before retrying
            continue
        
        # Prepare features
        train_x = prepare_features(rates_frame, n_features)
        
        if train_x.size > 0:
            # Make predictions
            predictions = loaded_model.predict(train_x)
            
            # Get the latest prediction
            latest_prediction = predictions[-1]
            
            # Check the number of open trades
            open_trades, open_positions = check_open_trades(symbol)
            if open_trades >= max_open_trades:
                logging.warning("Maximum number of open trades reached.")
                time.sleep(30)  # Wait for 1 minute before retrying
                continue
            
            # Check the account equity to ensure we haven't exceeded the max loss threshold
            account_info = mt5.account_info()
            if account_info.equity - account_info.balance <= max_loss_threshold:
                logging.warning("Maximum loss threshold exceeded. Halting trading.")
                break
            
            # Place a trade based on the latest prediction
            if latest_prediction == 1:
                result, trade_info = place_trade(symbol, "BUY")
            else:
                result, trade_info = place_trade(symbol, "SELL")
            
            if result.retcode == mt5.TRADE_RETCODE_DONE:
                open_trade_infos.append(trade_info)
        
        # Check for closed trades and update the CSV
        for trade_info in open_trade_infos[:]:
            positions = mt5.positions_get(symbol=symbol)
            if not positions or not any(pos.ticket == trade_info['retcode'] for pos in positions):
                history = mt5.history_deals_get(time_from=trade_info['time'], time_to=datetime.now())
                if history:
                    for deal in history:
                        if deal.symbol == symbol and deal.volume == trade_info['lot']:
                            trade_info['profit'] = deal.profit
                            trade_info['volume'] = deal.volume
                            log_trade_to_csv(trade_info)
                            open_trade_infos.remove(trade_info)
                            break
        
        # Wait for the next time interval
        time.sleep(30)  # Wait for 1 minute before fetching new data
    
    except Exception as e:
        logging.error(f"An error occurred: {e}")
        time.sleep(60)  # Wait for 1 minute before retrying

# Shutdown MetaTrader 5
mt5.shutdown()


C:\Users\ADMIN\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [15:35:56] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\gbm\gbtree.cc:388: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  warnings.warn(smsg, UserWarning)
C:\Users\ADMIN\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [15:35:56] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\context.cc:43: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
C:\Users\ADMIN\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [15:35:56] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


Unrealized P&L:  -60.0
Unrealized P&L:  -80.0
Unrealized P&L:  -120.0
Unrealized P&L:  -80.0


KeyboardInterrupt: 

In [8]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import joblib
import logging
import os
from threading import Thread, Event

# Configure logging
logging.basicConfig(filename='trading_new_log.log', level=logging.INFO, format='%(asctime)s %(message)s')

class TradingEngine:
    def __init__(self, symbol="EURUSD", timeframe=mt5.TIMEFRAME_M1, model_path="exodus.joblib", risk_percentage=1):
        self.symbol = symbol
        self.timeframe = timeframe
        self.model_path = model_path
        self.running = False
        self.thread = None
        self.stop_event = Event()
        
        # Risk management parameters
        self.daily_loss_limit = -500  # Daily loss limit
        self.trade_duration_limit = timedelta(hours=1)  # Maximum duration for open trades
        self.trailing_stop_threshold = 5  # Threshold to start trailing stop (in points)
        self.trailing_stop_distance = 3  # Distance to trail stop (in points)
        self.max_consecutive_losses = 3  # Maximum allowed consecutive losses
        self.current_consecutive_losses = 0  # Tracks current consecutive losses
        self.risk_percentage = risk_percentage  # Risk per trade as percentage of account balance
        
        # Trade and account tracking
        self.daily_loss = 0
        self.open_trade_infos = []
        self.start_of_day = datetime.now().date()
        
        # Load the model
        if os.path.exists(model_path):
            self.loaded_model = joblib.load(model_path)
            logging.info("Model loaded successfully")
        else:
            raise FileNotFoundError(f"Model file '{model_path}' not found.")

        # Initialize MetaTrader 5
        if not mt5.initialize():
            logging.error("MetaTrader 5 initialization failed")
            raise ConnectionError("Failed to initialize MetaTrader 5")

    def fetch_and_preprocess_data(self):
        rates = mt5.copy_rates_from_pos(self.symbol, self.timeframe, 0, 1000)
        rates_frame = pd.DataFrame(rates)
        rates_frame['time'] = pd.to_datetime(rates_frame['time'], unit='s')
        
        rates_frame['return'] = rates_frame['close'].diff()
        return_range = rates_frame['return'].max() - rates_frame['return'].min()
        rates_frame['return'] = rates_frame['return'] / return_range
        
        rates_frame['label'] = rates_frame['return'].shift(-1)
        rates_frame['label'] = rates_frame['label'].apply(lambda x: 1 if x > 0.0 else 0)
        rates_frame.dropna(inplace=True)
        
        return rates_frame

    # Function to prepare features In the TradingEngine class
    def prepare_features(self, rates_frame, n_features=60):
        train_x = np.empty((0, n_features))
        for i in range(n_features, len(rates_frame)):
            _x = rates_frame['return'].iloc[i-n_features:i].values.reshape(1, -1)
            train_x = np.vstack((train_x, _x))
        return train_x


    def calculate_lot_size(self):
        account_info = mt5.account_info()
        equity = account_info.equity
        risk_amount = (self.risk_percentage / 100) * equity
        return round(risk_amount / 100, 2)  # Simple calculation for lot size; refine as needed

    def log_trade_to_csv(self, trade_info):
        file_exists = os.path.isfile('trade_outcomes.csv')
        trade_info_df = pd.DataFrame([trade_info])
        trade_info_df.to_csv('trade_outcomes.csv', mode='a', header=not file_exists, index=False)

    def place_trade(self, action, slippage=5, stop_loss=15, take_profit=10):
        lot = self.calculate_lot_size()
        point = mt5.symbol_info(self.symbol).point
        price = mt5.symbol_info_tick(self.symbol).ask if action == "BUY" else mt5.symbol_info_tick(self.symbol).bid
        deviation = slippage
        
        # Calculate stop loss and take profit prices
        sl_price = price - stop_loss * point if action == "BUY" else price + stop_loss * point
        tp_price = price + take_profit * point if action == "BUY" else price - take_profit * point
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": self.symbol,
            "volume": lot,
            "type": mt5.ORDER_TYPE_BUY if action == "BUY" else mt5.ORDER_TYPE_SELL,
            "price": price,
            "sl": sl_price,
            "tp": tp_price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "Automated trade",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_RETURN,
        }
        
        result = mt5.order_send(request)
        trade_info = {
            'time': datetime.now(),
            'symbol': self.symbol,
            'action': action,
            'lot': lot,
            'price': price,
            'sl_price': sl_price,
            'tp_price': tp_price,
            'result': 'success' if result.retcode == mt5.TRADE_RETCODE_DONE else 'failure',
            'retcode': result.retcode,
            'profit': None,
            'volume': None
        }
        
        if result.retcode == mt5.TRADE_RETCODE_DONE:
            logging.info(f"Trade successful: {action} {lot} lot of {self.symbol} at {price}, SL={sl_price}, TP={tp_price}")
        else:
            logging.error(f"Trade failed, retcode={result.retcode}")
        
        return result, trade_info

    def trade_loop(self):
        while not self.stop_event.is_set():
            # Daily loss limit check
            if datetime.now().date() != self.start_of_day:
                self.start_of_day = datetime.now().date()
                self.daily_loss = 0

            if self.daily_loss <= self.daily_loss_limit:
                logging.warning("Daily loss limit reached. Halting trading for today.")
                break
            
            # Fetch and prepare data
            rates_frame = self.fetch_and_preprocess_data()
            if len(rates_frame) < 60:
                time.sleep(30)
                continue

            # In trade_loop
            predictions = self.loaded_model.predict(self.prepare_features(rates_frame, n_features=60))

            latest_prediction = predictions[-1]

            # Trade based on prediction
            if latest_prediction == 0:
                result, trade_info = self.place_trade("BUY")
            else:
                result, trade_info = self.place_trade("SELL")

            # Track trade results and update loss count
            if result.retcode != mt5.TRADE_RETCODE_DONE:
                self.current_consecutive_losses += 1
                if self.current_consecutive_losses >= self.max_consecutive_losses:
                    logging.warning("Max consecutive losses reached. Halting trading.")
                    break
            else:
                self.current_consecutive_losses = 0  # Reset on successful trade

            # Trailing stop logic can be implemented here...

            time.sleep(30)

        mt5.shutdown()
        logging.info("Trading engine stopped.")

    def start(self):
        if self.running:
            logging.warning("Trading engine is already running")
            return "Trading engine is already running"
        
        self.running = True
        self.stop_event.clear()
        self.thread = Thread(target=self.trade_loop)
        self.thread.start()
        return "Trading engine started"

    def stop(self):
        if not self.running:
            logging.warning("Trading engine is not running")
            return "Trading engine is not running"
        
        self.stop_event.set()
        self.thread.join()
        self.running = False
        return "Trading engine stopped"

# Example usage
trading_engine = TradingEngine()

# Start trading engine
print(trading_engine.start())

# Stop trading engine
# print(trading_engine.stop())


Trading engine started


C:\Users\ADMIN\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [15:47:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\gbm\gbtree.cc:388: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  warnings.warn(smsg, UserWarning)
C:\Users\ADMIN\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [15:47:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\context.cc:43: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
C:\Users\ADMIN\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [15:47:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


In [9]:

# Stop trading engine
print(trading_engine.stop())

Trading engine stopped


In [25]:
log_trade_to_csv(trade_info)

In [27]:
trade_info

{'time': datetime.datetime(2024, 10, 17, 11, 55, 37, 538),
 'symbol': 'EURUSD',
 'action': 'SELL',
 'lot': 10.0,
 'price': 1.08498,
 'sl_price': 1.08528,
 'tp_price': 1.08488,
 'result': 'success',
 'retcode': 10009,
 'profit': None,
 'volume': None}

In [ ]:
open_trade_infos

In [30]:
positions

(TradePosition(ticket=51709940717, time=1729165684, time_msc=1729165684513, time_update=1729165684, time_update_msc=1729165684513, type=0, magic=234000, identifier=51709940717, reason=3, volume=10.0, price_open=1.08514, sl=1.08484, tp=1.08524, price_current=1.08498, swap=0.0, profit=-160.0, symbol='EURUSD', comment='Automated trade', external_id=''),
 TradePosition(ticket=51709959914, time=1729166136, time_msc=1729166136763, time_update=1729166136, time_update_msc=1729166136763, type=1, magic=234000, identifier=51709959914, reason=3, volume=10.0, price_open=1.08498, sl=1.08528, tp=1.08488, price_current=1.08502, swap=0.0, profit=-40.0, symbol='EURUSD', comment='Automated trade', external_id=''))